In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv("../data/processed/cleaned_accident_data.csv")

In [3]:
df.info()
df.shape
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   accident_id        20000 non-null  int64  
 1   city               20000 non-null  object 
 2   state              20000 non-null  object 
 3   latitude           20000 non-null  float64
 4   longitude          20000 non-null  float64
 5   date               20000 non-null  object 
 6   time               20000 non-null  object 
 7   hour               20000 non-null  int64  
 8   day_of_week        20000 non-null  object 
 9   is_weekend         20000 non-null  int64  
 10  road_type          20000 non-null  object 
 11  lanes              20000 non-null  int64  
 12  traffic_signal     20000 non-null  int64  
 13  weather            20000 non-null  object 
 14  visibility         20000 non-null  object 
 15  temperature        20000 non-null  int64  
 16  traffic_density    200

,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,5:00,5,Sunday,1,...,low,32,high,weather,fatal,2,2,0,No Festival,0.85
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,4:00,4,Sunday,1,...,high,34,low,weather,major,4,3,0,No Festival,0.10
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10,13:00,13,Wednesday,0,...,low,21,medium,weather,minor,1,1,0,No Festival,0.45
3,3,Chandigarh,Punjab,30.787805,76.847507,2025-03-30,11:00,11,Sunday,1,...,low,30,high,distraction,minor,5,2,0,No Festival,0.65
4,4,Chennai,Tamil Nadu,12.965155,80.283313,2024-01-25,16:00,16,Thursday,0,...,high,24,low,distraction,minor,2,1,0,No Festival,0.10


# Feature Engineering

## Objective

The objective of this notebook is to prepare the cleaned dataset for machine learning by transforming variables, creating meaningful features, encoding categorical columns, and removing unnecessary information.

## Why is Feature Engineering Important?

Machine learning models cannot directly understand raw categorical data. Feature engineering improves data quality and creates meaningful representations that help models learn patterns more effectively.

In [4]:
df.drop(
    columns=[
        "accident_id",
        "latitude",
        "longitude"
    ],
    inplace=True
)

In [5]:
df.drop(
    columns=["date","time"],
    inplace=True
)

## Removing Unnecessary Features

Columns such as accident ID and geographical coordinates were removed because they do not contribute directly to predicting accident severity. Removing irrelevant features helps simplify the model and reduces unnecessary complexity.

In [6]:
df.dtypes

city                  object
state                 object
hour                   int64
day_of_week           object
is_weekend             int64
road_type             object
lanes                  int64
traffic_signal         int64
weather               object
visibility            object
temperature            int64
traffic_density       object
cause                 object
accident_severity     object
vehicles_involved      int64
casualties             int64
is_peak_hour           int64
festival              object
risk_score           float64
dtype: object

## Checking Data Types

The dataset contains both numerical and categorical variables. Categorical variables need to be converted into numerical representations before they can be used for machine learning.

# Identify Categorical Columns

In [7]:
categorical_columns = df.select_dtypes(include="object").columns

categorical_columns

Index(['city', 'state', 'day_of_week', 'road_type', 'weather', 'visibility',
       'traffic_density', 'cause', 'accident_severity', 'festival'],
      dtype='object')

In [8]:
target_encoder = LabelEncoder()

df["accident_severity"] = target_encoder.fit_transform(
    df["accident_severity"]
)

In [9]:
df["accident_severity"]

0        0
1        1
2        2
3        2
4        2
        ..
19995    2
19996    1
19997    2
19998    2
19999    2
Name: accident_severity, Length: 20000, dtype: int64

In [10]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()

df["accident_severity"] = target_encoder.fit_transform(
    df["accident_severity"]
)

In [11]:
label_encoders = {}

categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:

    le = LabelEncoder()

    df[column] = le.fit_transform(df[column])

    label_encoders[column] = le

In [12]:
df.head()

,city,state,hour,day_of_week,is_weekend,road_type,lanes,traffic_signal,weather,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,7,2,5,3,1,0,3,1,1,1,32,0,4,0,2,2,0,4,0.85
1,6,2,4,3,1,2,4,0,0,0,34,1,4,1,4,3,0,4,0.10
2,6,2,13,6,0,2,3,0,1,1,21,2,4,2,1,1,0,4,0.45
3,1,3,11,3,1,2,1,1,1,1,30,0,0,2,5,2,0,4,0.65
4,2,4,16,4,0,0,3,1,0,0,24,1,0,2,2,1,0,4,0.10


In [13]:
df.dtypes

city                   int64
state                  int64
hour                   int64
day_of_week            int64
is_weekend             int64
road_type              int64
lanes                  int64
traffic_signal         int64
weather                int64
visibility             int64
temperature            int64
traffic_density        int64
cause                  int64
accident_severity      int64
vehicles_involved      int64
casualties             int64
is_peak_hour           int64
festival               int64
risk_score           float64
dtype: object

# Encoding Categorical Features

All remaining categorical variables were converted into numerical values using Label Encoding. This transformation allows machine learning algorithms to process categorical information while preserving the original categories through encoder mappings.

# Separate Features (X) and Target (y)

In [14]:
X = df.drop(columns="accident_severity")

y = df["accident_severity"]

In [15]:
print("Features Shape :", X.shape)

print("Target Shape :", y.shape)

Features Shape : (20000, 18)
Target Shape : (20000,)


# Separating Features and Target

## Objective

Before training a machine learning model, the dataset must be divided into input features (`X`) and the target variable (`y`). The model learns patterns from the features to predict the target variable.

## Why is this important?

Separating features and the target ensures a clear machine learning workflow and prevents the model from using the answer itself during training.

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [18]:
print("Training Features :", X_train.shape)

print("Testing Features :", X_test.shape)

print("Training Target :", y_train.shape)

print("Testing Target :", y_test.shape)

Training Features : (16000, 18)
Testing Features : (4000, 18)
Training Target : (16000,)
Testing Target : (4000,)


In [19]:
df.to_csv(
    "../DATA/processed/ml_ready_accident_data.csv",
    index=False
)

print("ML Ready Dataset Saved Successfully!")

ML Ready Dataset Saved Successfully!


In [20]:
pip install joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [21]:
import joblib

In [23]:
joblib.dump(
    label_encoders,
    "../MODELS/label_encoders.pkl"
)
print("label_encoders Saved Successfully!")

label_encoders Saved Successfully!


In [24]:
joblib.dump(
    target_encoder,
    "../models/target_encoder.pkl"
)

['../models/target_encoder.pkl']